## Constructor Standings View

Ranks all constructors (teams) by season based on total points and wins.

---

#### Sources

| Table | Provides |
| --- | --- |
| `formula1.gold.facts_session_results` | Points, wins, podiums |
| `formula1.gold.dim_constructors` | Team name, nationality |

---

#### Logic

1. **Join** fact table with constructors dimension on `constructor_id`
2. **Aggregate** per team per season
3. **Rank** using `RANK()` by points then wins (descending)

---

#### Output - `formula1.gold.v_constructors_standings`

| Column | Description |
| --- | --- |
| `season` | Championship year |
| `constructor_id` | Team identifier |
| `constructor_name` | Team name |
| `nationality` | Team nationality |
| `race_starts` | Sessions entered |
| `total_points` | Points scored |
| `number_of_wins` | P1 finishes |
| `number_of_podiums` | Top-3 finishes |
| `standing` | Championship position |

In [0]:
CREATE or REPLACE view formula1.gold.v_constructors_standings
AS
WITH constructors_session_summary 
AS
(select 
 r.season,
 c.constructor_id,
 c.constructor_name,
 c.nationality,
 count(*) AS race_starts,
 sum(r.points) AS total_points,
 count_if(r.is_win) AS number_of_wins,
 count_if(r.is_podium) AS number_of_podiums
from formula1.gold.dim_constructors c
join formula1.gold.facts_session_results r
on 
 c.constructor_id = r.constructor_id
Group By 
 r.season,
 c.constructor_id,
 c.constructor_name,
 c.nationality)
SELECT season,
 constructor_id,
 constructor_name,
 nationality,
 race_starts,
 total_points,
 number_of_wins,
 number_of_podiums,
 RANK() OVER(PARTITION BY season ORDER BY total_points DESC, number_of_wins DESC) AS standing
from 
constructors_session_summary

In [0]:
select * from formula1.gold.v_constructors_standings where season = 2025;